# Taxa diária de vacinação de sarampo (1ª e 2ª dose) por mínimos quadrados — 2015 a 2026

Estima, para o Brasil, as taxas diárias $v_{1,g}$ e $v_{2,g}$ do modelo SEIHRVS com atrasos
(grupo $g=1$: 1–4 anos, $g=2$: 5–29 anos, $g=3$: 30–59 anos), ajustando o **número de doses mensais observado**
(SCR, soma das 27 UFs, **jan/2015 a set/2026**) ao **número de doses que o próprio sistema de EDAs prevê**.

## Como o modelo gera as doses

Doses aplicadas por dia, segundo as equações do modelo (1ª dose = ef + falha; $\Phi_{D2}$ sem o fator $e_2$,
que só decide quantos viram $V$):

| Grupo | 1ª dose / dia | 2ª dose / dia |
|---|---|---|
| 1 | $v_{1,1}\,S^{(1)}(t-\tau_{D1})\,e^{-\pi\tau_{D1}}$ | $v_{2,1}\,\Phi^{(1)}_{D1,falha}(t-\tau_{v2,1})\,e^{-\pi\tau_{v2,1}}$ |
| 2 | $v_{1,2}\,S^{(2)}(t)$ | $v_{2,2}\,\Phi^{(2)}_{D1,falha}(t-\tau_{v2,2})\,e^{-\pi\tau_{v2,2}}$ |
| 3 | $v_{1,3}\,S^{(3)}(t)$ | *(não há 2ª dose no modelo)* |

com $\Phi_{D1,falha}=(1-e_1)\times$(1ª dose/dia). $S^{(g)}$ é integrado junto (nascimentos $\Lambda$ em $S^{(1)}$,
mortalidade $\pi$, saída por vacinação). Como as doses dependem de $S^{(g)}$, que depende de $v_{1,g}$,
o ajuste de 1ª dose é **não linear**; a 2ª dose é ajustada depois, com $v_{1,g}$ fixo.

## Estimação

Cada taxa $v_{1,g}$ e $v_{2,g}$ é **constante dentro de cada regime** (dia$^{-1}$) e estimada por **mínimos quadrados não lineares**: minimiza-se a soma dos quadrados
dos resíduos entre as doses mensais observadas e as que o sistema de EDAs acima gera. Os regimes (`REGIMES`, editáveis) são **2015–2019**, **2020–2021** (pandemia) e **2022–2026**.
É só ajuste de parâmetros (sem previsão, sem sazonalidade, sem teste fora da amostra). A qualidade é medida pelo **erro percentual do total de doses por ano**
(modelo contra observado) e não pelo R² mensal, que fica próximo de zero para qualquer taxa constante. Para 2008–2022 usam-se **apenas os CSVs "sem campanhas"** (extração do DATASUS informada pelo autor).

## Hipóteses (declaradas, não medidas nos dados)

1. **Faixas dos arquivos** (informadas pelo autor): `faixa_etaria_1/2/3` = grupo 1 (1–4 anos, conforme o arquivo de população), grupo 2 (5–29) e grupo 3 (29/30–59). A 2ª dose do grupo 3 não existe no modelo e **não é usada**.
2. **2ª dose:** as planilhas trazem a 2ª dose *total*. No modelo, só o fluxo de falha vacinal ($1-e_1=7\%$ das 1ªs doses) recebe 2ª dose; por isso $v_{2,g}$ é ajustado a **7% das 2ªs doses observadas**. No grupo 1, a 2ª dose é SCR dose 2 + **tetraviral (SCRV) dose 1** (dose dos 15 meses); "-" na planilha vale 0.
3. **Sem infecção na dinâmica de $S$** ($\lambda=0$): aqui só importa o fluxo vacinal.
4. **População por faixa:** linha `Total` de `pop_brasil_2008-2025_grupo_etario_{1,2,3}.csv` (RIPSA/IBGE) em `ANO_POP` = **2015** (início do ajuste). Como $v_{1,g}\propto 1/S^{(g)}$, o nível de $v$ depende desse valor — veja a sensibilidade.
5. **Condição inicial:** $S^{(g)}$ no início da simulação = população do grupo × `FRAC_S0` (padrão 1: todo o grupo suscetível não vacinado). A simulação começa em jan/2015, sem história anterior: nos atrasos, $S^{(1)}(t-\tau_{D1})$ vale $S_0$ até jan/2016 (história constante do DDE) e o fluxo de falha anterior a 2015 é zero; por isso os primeiros meses de 2ª dose (3 no grupo 1, 1 no grupo 2) ficam fora do ajuste.
6. **Sem envelhecimento** entre grupos (o sistema de EDAs fornecido não o tem), logo $S^{(g)}$ não é reposto após 2015, exceto o grupo 1 pelos nascimentos.
7. **Nascimentos** ($\Lambda$): `nascimentos_mensais_brasil.csv` só cobre 2015+; os meses sem dado (a partir de abr/2026) usam o mesmo mês do ano disponível mais próximo (aproximação).
8. **Campanhas residuais:** os CSVs de 2008–2022 já vêm sem campanhas, mas o filtro automático continua ativo para o que restar (e para 2023+): mês cuja dose nacional excede `LIMIAR_CAMPANHA` (= 2,5) vezes a mediana móvel de 13 meses é **excluído do ajuste** (o modelo não representa campanhas) e listado. Esses meses também não retiram pessoas de $S$.
9. **Mês incompleto:** o último mês da planilha é excluído se estiver muito abaixo dos anteriores (detecção automática).
10. **Tetraviral antes de 2023:** a planilha de SCRV só cobre 2023+. Se a SCRV já era aplicada antes e não está nos CSVs de SCR 2008–2022, a 2ª dose do grupo 1 está **subestimada** nesse período. Para incluir, envie um CSV de SCRV 2008–2022 no mesmo formato, com nome contendo `tetraviral` e `2008_2022`.

**Como usar:** *Ambiente de execução → Executar tudo*; na célula [2.1] selecione de uma vez os arquivos:

| Arquivo | Qtde |
|---|---|
| `vacinacao_dose_{1,2}_grupo_etario_{g}_2008_2022_sem_campanhas.csv` (D1: g=1,2,3; D2: g=1,2) | 5 |
| `vacinacao-sarampo-dose1-grupo_etario{1,2,3}-2023a2026.xlsx` e `dose2-grupo_etario{1,2}` | 5 |
| `vacinacao-tetraviral-dose1-grupo_etario1-2023a202.xlsx` (SCRV, 2ª dose do grupo 1) | 1 |
| `nascimentos_mensais_brasil.csv` (`dados/`) | 1 |
| `pop_brasil_2008-2025_grupo_etario_{1,2,3}.csv` | 3 |

## Parte 1 — Parâmetros

In [ ]:
# [1.1] Imports e parâmetros do modelo
import os, re, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.signal import lfilter

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.6g}".format
BASE = Path("dados_vacinacao"); BASE.mkdir(exist_ok=True)

# --- Tabela de parâmetros do modelo (literatura/PNI) ---
PI = 1.8e-5                    # mortalidade natural diária
E1 = 0.93                      # eficácia da 1ª dose
E2 = 0.571                     # eficácia da 2ª dose (não entra no fluxo de doses aplicadas)
TAU_D1 = 365                   # idade da 1ª dose no grupo 1 (dias)
TAU_V2 = {1: 90, 2: 30}        # intervalo D1 -> D2 (dias)

# --- Hipóteses (ver introdução) ---
FRAC_S0     = {1: 1.0, 2: 1.0, 3: 1.0}   # fração do grupo inicialmente em S^(g)
POP_FAIXA   = None                       # opcional: {1: ..., 2: ..., 3: ...} para sobrescrever a população lida
FATOR_FALHA = 1 - E1                     # só a falha vacinal (7%) recebe 2ª dose no modelo
ANO_INICIO  = 2015                       # só dados a partir daqui (nascimentos existem desde jan/2015)
ANO_POP     = 2015                       # ano da população que define S^(g) inicial
LIMIAR_CAMPANHA = 2.5                    # mês/mediana móvel acima disso = campanha ou erro (fora do ajuste)
GRUPOS = {1: "1-4 anos", 2: "5-29 anos", 3: "30-59 anos"}
REGIMES = {"2015-2019": (2015, 2019), "2020-2021": (2020, 2021), "2022-2026": (2022, 2026)}   # um v constante por regime
NREG = len(REGIMES)
PARES = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2)]   # (dose, grupo) usados; sem D2 do grupo 3

## Parte 2 — Dados
### [2.1] Envio dos arquivos
*Fora do Colab*, defina `UPLOAD_DIRS` (pastas separadas por `;` no Windows ou `:` no Linux) com os arquivos.

In [ ]:
# [2.1] Upload (Colab) ou cópia local (UPLOAD_DIRS)
HIST = {(d, g): f"vacinacao_dose_{d}_grupo_etario_{g}_2008_2022_sem_campanhas.csv" for d, g in PARES}   # 2008-2022: só versão sem campanhas
NOVO = {(d, g): f"vacinacao-sarampo-dose{d}-grupo_etario{g}-2023a2026.xlsx" for d, g in PARES}
TETRA = "vacinacao-tetraviral-dose1-grupo_etario1-2023a202.xlsx"   # SCRV dose 1 = 2ª dose do grupo 1
POPS = {g: f"pop_brasil_2008-2025_grupo_etario_{g}.csv" for g in (1, 2, 3)}
ESPERADOS = list(HIST.values()) + list(NOVO.values()) + [TETRA, "nascimentos_mensais_brasil.csv"] + list(POPS.values())

def limpa(nome):                   # o Colab renomeia reenvios como 'arquivo (1).xlsx'
    return re.sub(r" \(\d+\)(?=\.[^.]+$)", "", nome)

def aceito(nome):
    return nome in ESPERADOS or ("tetraviral" in nome and "2008_2022" in nome)

try:
    from google.colab import files
    for nome, conteudo in files.upload().items():
        (BASE / limpa(nome)).write_bytes(conteudo)
        if nome != limpa(nome) or Path(nome).exists():
            Path(nome).unlink(missing_ok=True)
except ImportError:
    for pasta in os.environ["UPLOAD_DIRS"].split(os.pathsep):
        for arq in Path(pasta).iterdir():
            if aceito(arq.name):
                shutil.copy(arq, BASE / arq.name)

faltam = [n for n in ESPERADOS if not (BASE / n).exists()]
assert not faltam, f"Faltam arquivos: {faltam}"
TETRA_HIST = next(iter(sorted(BASE.glob("*tetraviral*2008_2022*.csv"))), None)
print("Arquivos obrigatórios OK:", len(ESPERADOS))
print("SCRV 2008-2022:", TETRA_HIST.name if TETRA_HIST else "NÃO enviada (2ª dose do grupo 1 pode estar subestimada antes de 2023)")

### [2.2] Leitura e junção 2008–2026
Os CSVs de 2008–2022 têm UF nas linhas e colunas `AAAA/Mes` (fora de ordem em alguns anos, por isso são lidas pelo nome); as planilhas
de 2023–2026 têm um bloco por ano. Soma-se as 27 UFs e junta-se as duas fontes numa série mensal contínua.

In [ ]:
# [2.2] Ler planilhas -> doses mensais do Brasil por (dose, grupo)
NOMES_MES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
MES_ABREV = {m.capitalize(): i + 1 for i, m in enumerate(NOMES_MES)}

def ler_historico(arquivo):
    df = pd.read_csv(BASE / arquivo, sep=";", encoding="latin-1")
    df = df[df.iloc[:, 0].notna()]
    df = df[df.iloc[:, 0].astype(str).str.strip().str.lower() != "total"]
    cols = [c for c in df.columns[1:] if "/" in c]
    per = [pd.Period(f"{c.split('/')[0].strip()}-{MES_ABREV[c.split('/')[1].strip()]:02d}", "M") for c in cols]
    vals = df[cols].replace("-", 0).apply(pd.to_numeric, errors="coerce")
    assert not vals.isna().any().any(), "valor nao numerico em " + arquivo
    assert vals.shape[0] == 27, "esperadas 27 UFs em " + arquivo
    vals.columns = per
    return vals.T.groupby(level=0).sum().T.sort_index(axis=1)

def ler_vacinacao(arquivo):
    df = pd.read_excel(BASE / arquivo, header=None)
    ano0 = int(df.iloc[1, 2])
    vals = df.iloc[3:, 2:].replace("-", 0).apply(pd.to_numeric, errors="coerce")   # "-" = zero doses
    meses = pd.period_range(f"{ano0}-01", periods=vals.shape[1], freq="M")
    nomes = df.iloc[2, 2:].astype(str).str.lower().tolist()
    assert nomes == [NOMES_MES[m.month - 1] for m in meses], "cabecalho de meses inesperado"
    vals.columns = meses
    assert vals.shape[0] == 27 and not vals.isna().any().any(), "esperadas 27 UFs sem vazios"
    return vals

def serie_nacional(par):
    h = ler_historico(HIST[par]).sum()
    n = ler_vacinacao(NOVO[par]).sum()
    s = pd.concat([h, n]).sort_index()
    assert not s.index.duplicated().any()
    assert (s.index == pd.period_range(s.index[0], s.index[-1], freq="M")).all(), "serie com buraco"
    return s

doses = pd.DataFrame({f"D{d}_G{g}": serie_nacional((d, g)) for d, g in PARES})
doses.index.name = "mes"

# 2ª dose do grupo 1 = SCR dose 2 + tetraviral (SCRV) dose 1 (dose dos 15 meses)
scrv = ler_vacinacao(TETRA).sum()
if TETRA_HIST is not None:
    scrv = pd.concat([ler_historico(TETRA_HIST.name).sum(), scrv]).sort_index()
scrv = scrv.reindex(doses.index).fillna(0.0)
doses["SCRV_G1"] = scrv.values
doses["D2_G1"] = doses["D2_G1"] + scrv.values
print(f"Grupo 1, 2ª dose: SCR {int(doses['D2_G1'].sum() - scrv.sum()):,} + SCRV {int(scrv.sum()):,} doses")

# Mês incompleto: total do último mês << média dos 3 anteriores
tot = doses.filter(regex="^D").sum(axis=1)
razao = tot.iloc[-1] / tot.iloc[-4:-1].mean()
parcial = razao < 0.7
print(f"Período: {doses.index[0]} a {doses.index[-1]}  ({len(doses)} meses)")
print(f"Último mês / média dos 3 anteriores = {razao:.2f} -> {'EXCLUÍDO (incompleto)' if parcial else 'mantido'}")
obs_df = (doses.iloc[:-1] if parcial else doses).loc[f"{ANO_INICIO}-01":]   # só a partir de ANO_INICIO
NM = len(obs_df)
ANO_INI, ANO_FIM = obs_df.index[0].year, obs_df.index[-1].year
NA = ANO_FIM - ANO_INI + 1
ANOS_OBS = np.array([p.year for p in obs_df.index])
print(f"Meses usados: {obs_df.index[0]} a {obs_df.index[-1]} ({NM}); anos: {ANO_INI}-{ANO_FIM}")

# Meses de campanha / erro de digitação: dose nacional >> mediana móvel de 13 meses
SERIES = [f"D{d}_G{g}" for d, g in PARES]
marcado = pd.DataFrame(index=obs_df.index, columns=SERIES, dtype=bool)
for col in SERIES:
    s = obs_df[col]
    marcado[col] = (s / s.rolling(13, center=True, min_periods=7).median()) > LIMIAR_CAMPANHA
print(f"\nMeses fora do ajuste (> {LIMIAR_CAMPANHA}x a mediana móvel):")
for col in SERIES:
    print(f"  {col}: {marcado[col].sum():2d} meses ->", ", ".join(str(p) for p in marcado.index[marcado[col]]))

# Totais anuais (milhares de doses) para conferência visual
anual = obs_df[SERIES + ["SCRV_G1"]].groupby(ANOS_OBS).sum() / 1e3
display(anual.round(0).astype(int))

### [2.3] Nascimentos ($\Lambda$) e população

In [ ]:
# [2.3] Nascimentos mensais e população por faixa
MESES_PT = {"janeiro": 1, "fevereiro": 2, "março": 3, "marco": 3, "abril": 4, "maio": 5, "junho": 6, "julho": 7,
            "agosto": 8, "setembro": 9, "outubro": 10, "novembro": 11, "dezembro": 12}
bruto = pd.read_csv(BASE / "nascimentos_mensais_brasil.csv", sep=";", encoding="utf-8")
nasc = {}
for rot, val in zip(bruto.iloc[:, 0].astype(str), bruto.iloc[:, 1]):
    m = re.match(r"^(\w+)/(\d{4})$", rot.strip())
    if m and m.group(1).lower() in MESES_PT:
        nasc[pd.Period(f"{m.group(2)}-{MESES_PT[m.group(1).lower()]:02d}", "M")] = float(val)
nasc = pd.Series(nasc).sort_index()
print("Nascimentos disponíveis:", nasc.index[0], "a", nasc.index[-1])

_cache_nasc = {}
def nascimentos_mes(p):
    # mês sem dado: usa o mesmo mês do ano disponível mais próximo (hipótese 7)
    if p not in _cache_nasc:
        if p in nasc.index:
            _cache_nasc[p] = nasc[p]
        else:
            q = min((q for q in nasc.index if q.month == p.month), key=lambda q: abs(q.year - p.year))
            _cache_nasc[p] = nasc[q]
    return _cache_nasc[p]

def ler_populacao(g):
    # arquivo do DATASUS/RIPSA em latin-1; cabeçalho na linha 5, linha 'Total' ao final da tabela
    df = pd.read_csv(BASE / POPS[g], sep=";", encoding="latin-1", skiprows=4, nrows=6)
    linha = df[df.iloc[:, 0].astype(str).str.strip().str.lower() == "total"]
    assert len(linha) == 1, "linha Total nao encontrada"
    return float(linha[str(ANO_POP)].iloc[0])

N_G = dict(POP_FAIXA) if POP_FAIXA else {g: ler_populacao(g) for g in (1, 2, 3)}
print(f"População por grupo etário, Brasil {ANO_POP}:")
print({GRUPOS[g]: f"{N_G[g]:,.0f}" for g in N_G})

## Parte 3 — Modelo direto e ajuste

### [3.1] Subsistema vacinal em passo diário
Integração (Euler, $\Delta t=1$ dia) de $S^{(g)}$ e dos fluxos de doses das equações do modelo, com $\lambda=0$ (vetorizada:
grupos 2 e 3 por produto acumulado; grupo 1 em blocos de $\tau_{D1}$ dias com `lfilter`). $v(t)$ é avaliado em cada dia pela forma paramétrica.

In [ ]:
# [3.1] Linha do tempo diária e funções do modelo direto
T0 = pd.Timestamp(f"{ANO_INI}-01-01")
FIM = obs_df.index[-1].to_timestamp(how="end").normalize()
HIST_DIAS = {1: 0, 2: 0, 3: 0}    # sem história antes de T0 (nascimentos só existem desde 2015)
POS_MES = {p: i for i, p in enumerate(obs_df.index)}
MES_INI_DIAS = np.array([(p.to_timestamp() - T0).days for p in obs_df.index])   # dia do início de cada mês

def reg_de_ano(ano):
    # índice do regime que contém o ano (fora do intervalo: regime mais próximo)
    for r, (ini, fim) in enumerate(REGIMES.values()):
        if ini <= ano <= fim:
            return r
    return 0 if ano < min(a for a, _ in REGIMES.values()) else NREG - 1

def linha_do_tempo(g):
    dias = pd.date_range(T0 - pd.Timedelta(days=HIST_DIAS[g]), FIM, freq="D")
    per = dias.to_period("M")
    mes_idx = np.array([POS_MES.get(p, -1) for p in per])
    lam = np.array([nascimentos_mes(p) / p.days_in_month for p in per])
    return dict(dias=dias, moy=dias.month.values - 1, tt=(dias - T0).days.values / 365.25,
                ano_idx=np.clip(dias.year.values - ANO_INI, 0, NA - 1),
                mes_idx=mes_idx, lam=lam, T=len(dias),
                reg_idx=np.array([reg_de_ano(y) for y in dias.year]))

TL = {g: linha_do_tempo(g) for g in (1, 2, 3)}
REG_MES = np.array([reg_de_ano(p.year) for p in obs_df.index])   # regime de cada mês

def simular_d1(g, v1, S0):
    # Doses de 1ª dose por dia e trajetória de S^(g)
    tl = TL[g]; T = tl["T"]; lam = tl["lam"]
    S = np.empty(T + 1); S[0] = S0
    if g != 1:
        S[1:] = S0 * np.cumprod(1.0 - v1 - PI)
        return v1 * S[:-1], S
    fl = np.zeros(T); atenua = np.exp(-PI * TAU_D1); a = 1.0 - PI
    for b in range(0, T, TAU_D1):
        i = np.arange(b, min(b + TAU_D1, T))
        fl[i] = v1[i] * S[np.maximum(i - TAU_D1, 0)] * atenua
        u = lam[i] - fl[i]
        S[i + 1], _ = lfilter([1.0], [1.0, -a], u, zi=[a * S[b]])
    return fl, S

def doses_d2(g, fl_d1, v2):
    tau = TAU_V2[g]
    falha = (1 - E1) * fl_d1
    atras = np.zeros_like(falha); atras[tau:] = falha[:-tau]
    return v2 * atras * np.exp(-PI * tau)

def mensal(g, x):
    # soma diária -> mensal (apenas meses usados no ajuste)
    idx = TL[g]["mes_idx"]; ok = idx >= 0
    return np.bincount(idx[ok], weights=x[ok], minlength=NM)

S0_BASE = {g: FRAC_S0[g] * N_G[g] for g in (1, 2, 3)}

### [3.2] Ajuste por mínimos quadrados não lineares (um parâmetro por regime)
Resíduo: (doses previstas − observadas) / média das observadas, por mês, **só nos meses válidos** (fora campanhas/erros); os parâmetros são ajustados em escala
normalizada (`s0`) com limite inferior zero. Reportam-se $R^2$ e RMSE mensais, o **erro percentual do total de doses por ano** (soma dos meses válidos) e o erro-padrão
aproximado de Gauss–Newton, $\hat\sigma^2(J^\top J)^{-1}$, com intervalo de 95% (±1,96·EP).

In [ ]:
# [3.2] Rotina de ajuste: um v constante por regime
def ajustar_v(obs, prever, s0, valido, k=NREG):
    esc = obs[valido].mean()
    res = lambda x: (prever(x * s0)[valido] - obs[valido]) / esc
    sol = least_squares(res, np.ones(k), bounds=(0.0, np.inf), method="trf")
    v = sol.x * s0
    n = int(valido.sum()); pred = prever(v)
    rss = float(np.sum((pred[valido] - obs[valido]) ** 2))
    tss = float(np.sum((obs[valido] - obs[valido].mean()) ** 2))
    cov = (np.sum(sol.fun ** 2) / max(n - k, 1)) * np.linalg.pinv(sol.jac.T @ sol.jac)
    return dict(v=v, se=np.sqrt(np.diag(cov)) * s0, n=n, rss=rss, r2=1 - rss / tss, rmse=np.sqrt(rss / n), pred=pred)

def erro_anual(obs, pred, valido):
    # erro % do total anual de doses (só meses válidos): (modelo - observado) / observado
    t = pd.DataFrame({"obs": obs * valido, "pred": pred * valido, "ano": ANOS_OBS}).groupby("ano").sum()
    return 100 * (t["pred"] - t["obs"]) / t["obs"]

### [3.3] 1ª dose — grupos 1, 2 e 3

In [ ]:
# [3.3] Ajuste da 1ª dose
ajustes_d1 = {}
VALIDO = {col: ~marcado[col].values for col in SERIES}
for g in (1, 2, 3):
    obs = obs_df[f"D1_G{g}"].values.astype(float)
    tl = TL[g]; valido = VALIDO[f"D1_G{g}"]
    prever = lambda v, g=g, tl=tl: mensal(g, simular_d1(g, v[tl["reg_idx"]], S0_BASE[g])[0])
    s0 = 1e-3 * obs[valido].mean() / prever(np.full(NREG, 1e-3))[valido].mean()
    ajustes_d1[g] = ajustar_v(obs, prever, s0, valido)
    a = ajustes_d1[g]
    print(f"1ª dose, grupo {g} ({GRUPOS[g]}): " + "  ".join(f"v[{nome}] = {v:.4g}" for nome, v in zip(REGIMES, a["v"])) + f"   R2 mensal = {a['r2']:.3f}")

### [3.4] 2ª dose — grupos 1 e 2
Com $v_{1,g}$ fixo nos valores estimados acima, a 2ª dose prevista é **linear** em $v_{2,g}$. O grupo 3 não tem 2ª dose no modelo (arquivo não usado).
O alvo do ajuste é **7% ($1-e_1$) das doses de 2ª dose observadas**: as planilhas são totais e o modelo só aplica 2ª dose ao fluxo de falha vacinal.

In [ ]:
# [3.4] Ajuste da 2ª dose
ajustes_d2, FL_D1 = {}, {}
for g in (1, 2):
    tl = TL[g]; valido = VALIDO[f"D2_G{g}"] & (MES_INI_DIAS >= TAU_V2[g])   # fluxo de falha anterior a T0 é desconhecido
    VALIDO[f"D2_G{g}"] = valido
    FL_D1[g] = simular_d1(g, ajustes_d1[g]["v"][tl["reg_idx"]], S0_BASE[g])[0]
    obs = FATOR_FALHA * obs_df[f"D2_G{g}"].values.astype(float)
    prever = lambda v, g=g, tl=tl: mensal(g, doses_d2(g, FL_D1[g], v[tl["reg_idx"]]))
    s0 = obs[valido].mean() / prever(np.ones(NREG))[valido].mean()
    ajustes_d2[g] = ajustar_v(obs, prever, s0, valido)
    a = ajustes_d2[g]
    print(f"2ª dose, grupo {g} ({GRUPOS[g]}): " + "  ".join(f"v[{nome}] = {v:.4g}" for nome, v in zip(REGIMES, a["v"])) + f"   R2 mensal = {a['r2']:.3f}")

## Parte 4 — Resultados
### [4.1] Taxas diárias estimadas e qualidade do ajuste
Unidades: $v_{1,g}$ em dia$^{-1}$ (fração de $S^{(g)}$ vacinada por dia); $v_{2,g}$ é o fator sobre o fluxo de falhas de 1ª dose, já ajustado ao fator de 7%
(a coluna `v_equiv_2a_dose_total` divide por 7%: seria o valor se o alvo fossem as 2ªs doses totais).

In [ ]:
# [4.1] Tabelas finais
linhas, qualidade, erros = [], [], {}
for dose, g in PARES:
    a = (ajustes_d1 if dose == 1 else ajustes_d2)[g]
    nome = f"v_{dose},{g}"
    valido = VALIDO[f"D{dose}_G{g}"]
    obs = (FATOR_FALHA if dose == 2 else 1.0) * obs_df[f"D{dose}_G{g}"].values.astype(float)
    for r, (reg, v, se) in enumerate(zip(REGIMES, a["v"], a["se"])):
        row = dict(parametro=nome, faixa=GRUPOS[g], regime=reg, v=v, erro_padrao=se,
                   ic95_inf=v - 1.96 * se, ic95_sup=v + 1.96 * se, meses_ajuste=int((valido & (REG_MES == r)).sum()))
        if dose == 2:
            row["v_equiv_2a_dose_total"] = v / FATOR_FALHA
        linhas.append(row)
    ea = erro_anual(obs, a["pred"], valido)
    erros[nome] = ea
    qualidade.append(dict(parametro=nome, faixa=GRUPOS[g], erro_total_pct=100 * (a["pred"][valido].sum() - obs[valido].sum()) / obs[valido].sum(),
                          erro_abs_medio_anual_pct=ea.abs().mean(), erro_abs_max_anual_pct=ea.abs().max(),
                          R2_mensal=a["r2"], RMSE_mensal=a["rmse"], meses_ajuste=a["n"], meses_excluidos=int(marcado[f"D{dose}_G{g}"].sum())))
tabela_v = pd.DataFrame(linhas).set_index(["parametro", "regime"])
tabela_q = pd.DataFrame(qualidade).set_index("parametro")
tabela_e = pd.DataFrame(erros); tabela_e.index.name = "ano"
display(tabela_v)
print("\nQualidade do ajuste:")
display(tabela_q)
print("\nErro % do total anual de doses (modelo - observado) / observado:")
display(tabela_e.round(1))

> **Nota sobre $v_{2,g}$.** O ajuste usa 7% ($1-e_1$) das 2ªs doses observadas, que é o que o modelo atribui ao fluxo de falha vacinal.
> Logo $v_{2,g}$ (do modelo) = 7% do valor que se obteria ajustando as 2ªs doses totais; a coluna `v_equiv_2a_dose_total` mostra esse valor
> total (= $v_{2,g}/(1-e_1)$). Os erros percentuais são os mesmos nas duas escalas.

### [4.2] Ajuste aos dados

In [ ]:
# [4.2] Doses mensais observadas x modelo (v constante por regime)
fig, ax = plt.subplots(1, 5, figsize=(24, 3.8))
x = obs_df.index.to_timestamp()
for j, (dose, g) in enumerate(PARES):
    a = (ajustes_d1 if dose == 1 else ajustes_d2)[g]
    esc = FATOR_FALHA if dose == 2 else 1.0
    obs = esc * obs_df[f"D{dose}_G{g}"].values
    ok = VALIDO[f"D{dose}_G{g}"]
    ax[j].plot(x[ok], obs[ok], ".", ms=4, label="observado" + (" x 7%" if dose == 2 else ""))
    ax[j].plot(x[~ok], obs[~ok], "x", color="tab:red", ms=5, label="fora do ajuste")
    ax[j].plot(x, a["pred"], "-", lw=1.2, label="modelo")
    for ano in (2020, 2022):
        ax[j].axvline(pd.Timestamp(f"{ano}-01-01"), color="gray", ls=":", lw=0.8)
    ax[j].set_ylim(0, np.nanpercentile(obs[ok], 99) * 1.6)
    ax[j].set_title(f"D{dose}, {GRUPOS[g]}"); ax[j].legend(fontsize=6); ax[j].tick_params(axis="x", rotation=45)
    ax[j].set_ylabel("doses/mês")
plt.tight_layout(); plt.show()

### [4.3] Sensibilidade à população do grupo
$v_{1,g}$ é praticamente inversamente proporcional a $S^{(g)}$, que é uma hipótese (população de `ANO_POP` e `FRAC_S0`). Reajusta-se $v_{1,g}$ com $S^{(g)}_0$ multiplicado por 0,8 / 1 / 1,2.

In [ ]:
# [4.3] Sensibilidade do v_1,g a S0
linhas = []
for g in (1, 2, 3):
    obs = obs_df[f"D1_G{g}"].values.astype(float); tl = TL[g]; valido = VALIDO[f"D1_G{g}"]
    for fator in (0.8, 1.0, 1.2):
        S0 = S0_BASE[g] * fator
        prever = lambda v: mensal(g, simular_d1(g, v[tl["reg_idx"]], S0)[0])
        s0 = 1e-3 * obs[valido].mean() / prever(np.full(NREG, 1e-3))[valido].mean()
        a = ajustar_v(obs, prever, s0, valido)
        for reg, v in zip(REGIMES, a["v"]):
            linhas.append(dict(grupo=GRUPOS[g], regime=reg, fator_S0=fator, v_1=v))
sens = pd.DataFrame(linhas)
display(sens.pivot(index=["grupo", "regime"], columns="fator_S0", values="v_1"))

### [4.4] Exportar

In [ ]:
# [4.4] Exportar resultados
excluidos = marcado.stack().loc[lambda s: s].reset_index()
excluidos.columns = ["mes", "serie", "marcado"]
excluidos["mes"] = excluidos["mes"].astype(str)
saidas = {"taxas_vacinacao_sarampo_mq.csv": tabela_v.reset_index(),
          "qualidade_ajuste_sarampo_mq.csv": tabela_q.reset_index(),
          "erro_anual_sarampo_mq.csv": tabela_e.reset_index(),
          "doses_mensais_sarampo_brasil.csv": doses.reset_index().astype({"mes": str}),
          "meses_excluidos_campanha.csv": excluidos.drop(columns="marcado"),
          "sensibilidade_S0.csv": sens}
for nome, df in saidas.items():
    df.to_csv(nome, sep=";", index=False, encoding="utf-8-sig")
    print("gravado:", Path(nome).resolve())
try:
    from google.colab import files
    for nome in saidas:
        files.download(nome)
except ImportError:
    pass